In [10]:
import easyocr
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import time
from PIL import Image
from typing import List, Dict, Optional
import re
import json


In [11]:
def perform_ocr(image_path):
    # Initialize the OCR reader
    reader = easyocr.Reader(['en'], gpu=False) 
    
    # Read the image
    start_time = time.time()
    
    # Check if the image exists
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
        return None
    
    # Read and process the image
    results = reader.readtext(image_path, detail=0, paragraph=True)
    
    # Calculate processing time
    processing_time = time.time() - start_time
    print(f"OCR completed in {processing_time:.2f} seconds")
    
    return results


In [12]:
def visualize_results(image_path, results):
    
    if results is None:
        print("No results to visualize")
        return
    
    # Read the image with OpenCV
    image = cv2.imread(image_path)
    
    # Convert from BGR to RGB for matplotlib
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    # Draw bounding boxes and text
    for (bbox, text, prob) in results:
        # Bounding box coordinates
        (top_left, top_right, bottom_right, bottom_left) = bbox
        top_left = tuple(map(int, top_left))
        bottom_right = tuple(map(int, bottom_right))
        
        # Draw the bounding box
        cv2.rectangle(image, top_left, bottom_right, (0, 255, 0), 2)
        
        # Draw the text
        cv2.putText(image, text, (top_left[0], top_left[1] - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        
        # Print the text and confidence
        print(f"Text: {text} (Confidence: {prob:.2f})")
    
    # Display the image with bounding boxes
    plt.figure(figsize=(10, 10))
    plt.imshow(image)
    plt.axis('off')
    plt.title('OCR Results')
    plt.show()


In [13]:
def process_directory(directory_path, languages=['en'], gpu=False):
    # Supported image extensions
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']
    
    # Process each file in the directory
    for filename in os.listdir(directory_path):
        file_path = os.path.join(directory_path, filename)
        
        # Check if it's a file and has a valid extension
        if os.path.isfile(file_path) and any(filename.lower().endswith(ext) for ext in valid_extensions):
            print(f"\nProcessing {filename}...")
            results = perform_ocr(file_path, languages, gpu)
            
            if results:
                print(f"Found {len(results)} text regions in {filename}")
                visualize_results(file_path, results)


In [14]:
# Configuration - set these variables directly
image_path = "./data/test3.png" 

# Main execution
if image_path:
    # Process the single image
    text = perform_ocr(image_path)
    print(text)

Using CPU. Note: This module is much faster with a GPU.
c:\Users\STORM Tech\Desktop\PDFMarker\venv\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


OCR completed in 1.92 seconds
['It was the best of times, it was the worst of times, it was the age of wisdom; it was the age of foolishness .']


In [15]:
def find_all_sentence_contexts(
    full_text: str,
    search_terms: List[str],
    language: str = 'fr'
) -> Dict[str, List[Dict[str, Optional[str]]]]:
    
    # Basic sentence splitting regex (works for French and English)
    sentence_endings = r'(?<!\w\.\w.)(?<![A-ZÀ-Ü][a-zà-ü]\.)(?<=\.|\?|\!|\…|\n)\s+'
    sentences = [s.strip() for s in re.split(sentence_endings, full_text) if s.strip()]
    
    # Precompile case-insensitive regex patterns for each term
    patterns = {
        term: re.compile(rf'(?<!\w){re.escape(term)}(?!\w)', re.IGNORECASE)
        for term in search_terms
    }
    
    results = {term: [] for term in search_terms}
    
    for idx, sentence in enumerate(sentences):
        for term, pattern in patterns.items():
            if pattern.search(sentence):
                context = {
                    'prev': sentences[idx-1] if idx > 0 else None,
                    'match': sentence,
                    'next': sentences[idx+1] if idx < len(sentences)-1 else None
                }
                results[term].append(context)
    
    return results

In [ ]:
terms = ["the"] 
contexts = find_all_sentence_contexts(text[0], terms)
# with open('./output/test.json', 'w', encoding='utf-8') as f:
#         json.dump(contexts, f, ensure_ascii=False, indent=2)
# print(f"Contexts saved")


Contexts saved


In [ ]:
# for term, occurrences in contexts.items():
#     print(f"\n{'='*80}")
#     print(f"Term: '{term}' - Found {len(occurrences)} occurrence(s)")
#     print(f"{'='*80}")
    
#     for i, ctx in enumerate(occurrences, 1):
#         print(f"\nContext {i}:")
#         if ctx['prev']:
#             print(f"[Previous] {ctx['prev']}")
#         print(f"[Match]    {ctx['match']}")
#         if ctx['next']:
#             print(f"[Next]     {ctx['next']}")
#         print("-" * 40)


Term: 'the' - Found 1 occurrence(s)

Context 1:
[Match]    It was the best of times, it was the worst of times, it was the age of wisdom; it was the age of foolishness .
----------------------------------------
